# Lab 1 — Repaso Práctico de Spark con continuidad BDCC → ADBD
**Análisis de Big Data · Magíster en Data Science UDD · Sesión 1**

**Estudiante:** Dante Aguirre Bravo

**Docente:** Luis Castillo Faune

**Objetivo:** resolver el Lab 1 conectando explícitamente cada respuesta con los contenidos y experiencias del curso previo **Big Data y Cloud Computing (BDCC)**: arquitectura Medallion, formatos columnares, DAG y evaluación *lazy*, transformaciones *narrow/wide*, *shuffle*, Spark SQL, Catalyst, particionamiento, caching estratégico y FinOps.

**Entorno:** Google Colab. Ejecutar las celdas en orden o usar **Entorno de ejecución → Ejecutar todas**.

**Dataset:** NYC Yellow Taxi — enero 2025 (~3 millones de viajes, Parquet público de NYC TLC).


## Continuidad con el semestre anterior

Este laboratorio no parte desde cero. Retoma contenidos trabajados con el mismo profesor en **Big Data y Cloud Computing** y los aplica a una versión más reciente del dataset NYC Taxi.

| Sección del Lab 1 | Contenido previo relacionado | Evidencia del curso anterior |
|---|---|---|
| Parquet vs. CSV | Formatos fila/columna y arquitectura Medallion | Clase 2, págs. 6 y 8; laboratorio Parquet→Delta |
| Transformaciones y acciones | DAG, evaluación *lazy* y linaje | Clase 3, págs. 7–8; pipeline Bronze→Silver |
| Cache | Caching estratégico y FinOps | Clases 5 y 6; memoria entendida como inversión |
| `GROUP BY` y `Exchange` | Transformaciones *wide* y *shuffle* | Clase 3, págs. 9–12; medición empírica del laboratorio anterior |
| Spark SQL | Catalyst Optimizer | Clase 3, pág. 12 |
| Particiones | Regla de oro y *partition pruning* | Clases 5–6; benchmark del proyecto El Monte |
| Calidad y sesgo | Bronze → Silver → Gold | Clase 2 y pipeline territorial de El Monte |

> La conexión principal es práctica: en el laboratorio previo se leyó NYC Taxi 2023 como capa Bronze, se construyó una capa Silver y se identificaron nodos `Exchange` con `.explain(True)`. Aquí se repite ese razonamiento con los viajes de enero de 2025.


## 0. Setup

In [ ]:
# Solo en Colab / Jupyter local
%pip install -q pyspark==3.5.4


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (SparkSession.builder
         .appName("ADBD-Lab1-Repaso")
         .master("local[2]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())
spark.version


In [ ]:
# Descarga del dataset (~50 MB). En Databricks CE usar %sh o subir a /FileStore.
import urllib.request, os
URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
PATH = "yellow_tripdata_2025-01.parquet"
if not os.path.exists(PATH):
    urllib.request.urlretrieve(URL, PATH)
print(os.path.getsize(PATH) / 1e6, "MB")


## 1. Repaso: Parquet y esquemas

Parquet es columnar y autodescriptivo: trae el esquema consigo y estadísticas por bloque (lo vimos en el módulo 2 de BDCC).

In [ ]:
df = spark.read.parquet(PATH)
df.printSchema()


In [ ]:
df.select("tpep_pickup_datetime", "PULocationID", "trip_distance",
          "payment_type", "fare_amount", "tip_amount", "total_amount").show(5)


**Pregunta de repaso 1:** `df.count()` sobre este Parquet es casi instantáneo. ¿Qué habría hecho Spark si el archivo fuera un CSV de 50 MB? ¿Por qué la diferencia?

## Respuesta

Si el archivo fuera un CSV de 50 MB, al ejecutar `df.count()` Spark tendría que leer y procesar todas las líneas para determinar la cantidad total de registros. También tendría que interpretar los separadores, las comillas y las columnas. Si se utiliza `inferSchema`, además debe revisar los valores para identificar sus tipos.

En cambio, Parquet es un formato columnar y autodescriptivo. Almacena el esquema y estadísticas en sus metadatos, lo que permite que Spark reduzca la cantidad de información que necesita leer. Catalyst también puede aplicar optimizaciones como la poda de columnas para evitar procesar datos que no son necesarios.

Esto se relaciona con lo trabajado en BDCC, donde Parquet se utilizó en las capas Bronze y Silver porque es más eficiente para consultas analíticas que un formato de texto como CSV. La diferencia se vuelve todavía más importante cuando los datos se almacenan y procesan de forma distribuida en la nube.

En resumen, con CSV Spark debe leer y procesar el contenido como texto, mientras que con Parquet puede aprovechar el esquema, los metadatos y su organización columnar para realizar la operación de forma más eficiente.


## 2. Repaso: transformaciones vs acciones (evaluación perezosa)

La idea más importante de BDCC: las transformaciones **no ejecutan nada**; construyen un plan. Solo una **acción** dispara el job.

In [ ]:
import time

# Esto NO ejecuta ningún job (verifícalo: es instantáneo)
t0 = time.time()
limpio = (df
    .filter((F.col("total_amount") > 0) & (F.col("trip_distance") > 0))
    # Intervalo semiabierto: incluye todo el 31 de enero y excluye febrero.
    .filter((F.col("tpep_pickup_datetime") >= F.lit("2025-01-01")) &
            (F.col("tpep_pickup_datetime") < F.lit("2025-02-01")))
    .withColumn("fecha", F.to_date("tpep_pickup_datetime"))
    .withColumn("hora", F.hour("tpep_pickup_datetime"))
    .withColumn("forma_pago",
        F.when(F.col("payment_type") == 1, "tarjeta")
         .when(F.col("payment_type") == 2, "efectivo")
         .otherwise("otro")))
print(f"Definir transformaciones: {time.time()-t0:.3f} s  (no se ejecutó nada)")




In [ ]:
# Esto SÍ ejecuta (count es una acción)
t0 = time.time()
n = limpio.count()
print(f"count() = {n:,} filas en {time.time()-t0:.2f} s  (aquí corrió el job)")

**Pregunta de repaso 2:** ¿cuántos *jobs* lanzó la celda anterior? ¿Y si volvemos a llamar `limpio.count()`, Spark recalcula todo o recuerda el resultado?

## Respuesta

La celda anterior lanzó **un job**, porque `limpio.count()` es una acción que obliga a Spark a ejecutar las transformaciones que estaban pendientes en el DAG.

Las operaciones utilizadas para construir `limpio`, como `filter` y `withColumn`, son transformaciones *narrow*. Estas operaciones no procesan los datos inmediatamente, sino que agregan pasos al plan de ejecución. El procesamiento comienza cuando se llama a una acción como `count()`.

Si volvemos a ejecutar `limpio.count()` antes de utilizar caché, Spark realizará nuevamente el proceso desde el archivo Parquet. Spark conserva el linaje que indica cómo calcular el DataFrame, pero no guarda automáticamente el resultado de la acción anterior. Una segunda ejecución podría ser más rápida por la caché del sistema operativo, aunque eso no significa que el DataFrame esté almacenado por Spark.

Para evitar este nuevo cálculo sería necesario utilizar `cache()` o `persist()` y materializar el DataFrame mediante una acción.

En resumen, `count()` lanzó un job y, si se ejecuta nuevamente sin persistir `limpio`, Spark volverá a recorrer su linaje y realizará el cálculo otra vez.


## 3. Repaso: caching con criterio

`cache()` guarda el DataFrame para reutilizarlo la próxima vez que se materializa. En este laboratorio usamos `persist(StorageLevel.DISK_ONLY)` porque el dataset completo puede superar la memoria disponible en una ejecución local.


In [ ]:
from pyspark import StorageLevel

spark.catalog.clearCache()
limpio = limpio.select(
    "tpep_pickup_datetime", "fecha", "hora", "forma_pago",
    "total_amount", "tip_amount"
 ).coalesce(2)
t0 = time.time(); limpio.groupBy("forma_pago").count().collect()
print(f"1a agregación (sin cache): {time.time()-t0:.2f} s")

limpio.persist(StorageLevel.DISK_ONLY)
limpio.count()   # materializa el cache

t0 = time.time(); limpio.groupBy("forma_pago").count().collect()
print(f"2a agregación (con cache): {time.time()-t0:.2f} s")


**Pregunta de repaso 3:** ¿en qué casos `cache()` puede *empeorar* las cosas? (Pista: memoria de los executors y datasets que se usan una sola vez.)

## Respuesta

El uso de `cache()` puede empeorar el rendimiento cuando el DataFrame se utiliza una sola vez, porque Spark debe ocupar tiempo y recursos para almacenarlo sin obtener un beneficio posterior.

También puede ser perjudicial cuando el dataset es demasiado grande para la memoria disponible en los *executors*. En ese caso, Spark puede expulsar otros datos de la memoria, almacenar parte de la información en disco, volver a calcular particiones o dedicar más recursos a la administración de memoria.

Tampoco conviene mantener un DataFrame en caché después de terminar de utilizarlo ni guardar una versión con muchas columnas cuando solamente se necesita un subconjunto. En esos casos se puede liberar la memoria con `unpersist()`.

En este laboratorio, cachear `limpio` puede ser útil porque se reutiliza en varias agregaciones y consultas. Esto se relaciona con lo revisado en BDCC: la memoria debe utilizarse como un recurso limitado y solo conviene persistir un resultado cuando el ahorro de nuevos cálculos compensa el costo de almacenarlo.

En resumen, `cache()` puede empeorar el rendimiento cuando el DataFrame se usa una sola vez, ocupa demasiada memoria, es barato de recalcular o permanece almacenado más tiempo del necesario.


## 4. Repaso: Spark SQL

Mismo motor, dos interfaces: API de DataFrames o SQL sobre una vista temporal. Como vimos en la **Clase 3, pág. 12**, ambas interfaces llegan al mismo plan porque **Catalyst Optimizer** transforma el plan lógico, lo optimiza y selecciona el plan físico. La elección entre SQL y DataFrames responde principalmente a legibilidad y mantenibilidad, no a que una interfaz sea intrínsecamente más rápida.


In [ ]:
limpio.createOrReplaceTempView("viajes")

spark.sql("""
SELECT forma_pago,
       COUNT(*)                     AS viajes,
       ROUND(AVG(total_amount), 2)  AS ticket_promedio,
       ROUND(AVG(tip_amount), 2)    AS propina_promedio
FROM viajes
GROUP BY forma_pago
ORDER BY viajes DESC
""").show()


In [ ]:
# La misma consulta con la API de DataFrames — produce el MISMO plan
(limpio.groupBy("forma_pago")
       .agg(F.count("*").alias("viajes"),
            F.round(F.avg("total_amount"), 2).alias("ticket_promedio"),
            F.round(F.avg("tip_amount"), 2).alias("propina_promedio"))
       .orderBy(F.desc("viajes"))
       .show())


**Observa:** ¿la propina promedio en efectivo es real o un artefacto de los datos? Las propinas en efectivo rara vez se registran. Este es el primer ejemplo de *sesgo de registro*, tema que se estudiará en este curso.

## Respuesta

La propina promedio asociada a los pagos en efectivo probablemente corresponde a un **artefacto de los datos** y no representa el valor real de las propinas entregadas por los pasajeros.

Esto ocurre porque las propinas pagadas con tarjeta quedan registradas en el sistema, mientras que las propinas entregadas en efectivo normalmente no se registran. Por lo tanto, un valor igual a cero puede significar que no hubo propina, pero también puede significar que la propina existió y el sistema no la observó.

Este problema corresponde a un sesgo de registro. Aunque una capa Silver puede corregir tipos, filtrar valores inválidos y controlar datos nulos, no puede recuperar información que nunca fue capturada en la fuente original. Por esta razón, no sería correcto concluir que los pasajeros que pagan en efectivo entregan menos propina basándose únicamente en este promedio.

En resumen, la propina promedio en efectivo está subestimada y debe interpretarse como propina registrada, no necesariamente como la propina real entregada al conductor.


## 5. Vistazo al plan de ejecución

En BDCC diagnosticábamos con la Spark UI; `explain()` muestra el mismo plan en texto. La próxima sesión aprenderemos a leerlo en detalle.

In [ ]:
spark.sql("""
SELECT forma_pago, fecha, COUNT(*) AS viajes
FROM viajes
GROUP BY forma_pago, fecha
""").explain()


**Pregunta de repaso 4:** busca `Exchange` en el plan, que representa al shuffle. ¿Por qué un `GROUP BY` lo necesita?

## Respuesta

Antes de ejecutar el `GROUP BY`, las filas con la misma combinación de `forma_pago` y `fecha` pueden estar distribuidas en diferentes particiones. Para obtener un conteo global correcto, Spark necesita reunir los resultados que pertenecen a la misma clave.

Primero, cada partición puede calcular conteos parciales. Luego aparece `Exchange hashpartitioning(...)`, que redistribuye esos resultados para que todas las claves iguales lleguen a la misma partición. Este movimiento de datos entre particiones corresponde al *shuffle*.

El `GROUP BY` es una transformación *wide* porque la salida de una partición puede depender de datos que estaban ubicados en varias particiones de entrada. A diferencia de operaciones como `filter` o `select`, este proceso puede requerir serialización, escritura en disco y transferencia de datos entre executors. Por eso suele ser una de las operaciones más costosas en un entorno distribuido.

La configuración `spark.sql.shuffle.partitions = 8` indica que Spark utilizará ocho particiones para esta redistribución en el laboratorio. Este valor es razonable para una ejecución local, pero en un clúster debe ajustarse de acuerdo con el volumen de datos y los recursos disponibles.

En resumen, `GROUP BY` necesita un `Exchange` porque Spark debe redistribuir los datos para reunir las claves iguales y calcular correctamente cada grupo.


## 6. Cierre y verificación

Las cuatro preguntas de repaso y la observación sobre sesgo de registro quedaron respondidas en sus secciones correspondientes. La siguiente celda resume el conjunto limpio y libera la caché al finalizar.


In [ ]:
# Comprobaciones finales de calidad del subconjunto analizado
(limpio.agg(
    F.count("*").alias("filas_limpias"),
    F.min("tpep_pickup_datetime").alias("primer_pickup"),
    F.max("tpep_pickup_datetime").alias("ultimo_pickup"),
    F.round(F.avg("total_amount"), 2).alias("monto_total_promedio")
).show(truncate=False))

limpio.unpersist()
spark.stop()


## Fuentes internas revisadas

- `Big Data Curso previo /Clase 2.pdf`: arquitectura Medallion y formatos columnares (págs. 6 y 8).
- `Big Data Curso previo /Clase 3.pdf`: DAG, evaluación *lazy*, *narrow/wide*, shuffle, Spark SQL y Catalyst (págs. 7–12).
- `Big Data Curso previo /Clase 5 Mini.pdf`: plan físico, particionamiento, caching estratégico y FinOps (págs. 3–9).
- `Big Data Curso previo /Clase 6.pdf`: Spark UI, shuffle, reglas heurísticas de particionamiento y caché (págs. 2–8).
- `Big Data Curso previo /Lab_DeParquetADeltaLake_dab.ipynb`: experiencia previa con NYC Taxi 2023, Bronze→Silver, Delta Lake, `explain(True)` y medición Narrow/Wide.
- `Big Data Curso previo /PROYECTO/fase2_gestion_territorial_el_monte_grupo_2`: pipeline Medallion, benchmark de Parquet particionado, ocho particiones de shuffle para ejecución local y evidencia FinOps.

### Síntesis de continuidad

El semestre anterior se aprendió a construir y operar pipelines distribuidos; este laboratorio vuelve sobre esas bases para comenzar el análisis crítico. El paso de BDCC a ADBD no cambia el motor: cambia la pregunta. Ya no basta con que el pipeline sea correcto y eficiente; también debemos examinar qué representa el dato, qué no registra y qué sesgos puede transmitir al análisis.




---
**Entrega:** notebook ejecutado (con outputs visibles) al repositorio del curso antes de la Sesión 2 (viernes 14 de agosto 18:30).
